In [1]:
from pathlib import Path
import sys
import pandas as pd
import joblib
import numpy as np


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from fetch_data import (
    download_nasa_stellar,
    extract_gaia_ids,
    fetch_gaia,
    build_labeled_training_dataset,
)

from exoplanet_features import prepare_model_input

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).


In [2]:

xgb_pipeline = joblib.load(
    PROJECT_ROOT/'model'/'xgboost_exoplanet_model.pkl'
)

In [3]:
def probability_label(prob):
    if prob >= 0.9:
        return "Very High Chance"
    elif prob >= 0.7:
        return "High Chance"
    elif prob >= 0.5:
        return "Possible Candidate"
    elif prob >= 0.3:
        return "Low Probability"
    else:
        return "Unlikely Host"

In [4]:
def find_top_gaia_candidates(
    model_pipeline,
    fetch_function,
    prepare_model_input,
    n_iterations=10,
    n_stars_per_batch=100_000,
    top_n=100,
    random_index_start=1_000_000,
    random_index_step=1_000_000,
    output_path=None
):
    top_candidates = pd.DataFrame()
    seen_ids = set()

    for i in range(n_iterations):
        random_min = random_index_start + i * random_index_step
        random_max = random_min + random_index_step

        print("=" * 60)
        print(f"Batch {i + 1}/{n_iterations}")
        print(f"Random index range: {random_min} - {random_max}")
        print("=" * 60)

        batch_df = fetch_function(
            mode="prediction",
            n_stars=n_stars_per_batch,
            random_index_min=random_min,
            random_index_max=random_max,
            output_path=None
        )

        if batch_df is None or batch_df.empty:
            print("No data fetched. Skipping batch.")
            continue

        batch_df = batch_df[
            ~batch_df["gaia_dr3_id"].isin(seen_ids)
        ].copy()

        if batch_df.empty:
            print("All stars already processed. Skipping.")
            continue

        seen_ids.update(batch_df["gaia_dr3_id"].tolist())

        X_batch = prepare_model_input(batch_df)

        batch_df["host_probability"] = model_pipeline.predict_proba(X_batch)[:, 1]

        batch_df["candidate_label"] = batch_df["host_probability"].apply(
            probability_label
        )

        batch_top = batch_df.sort_values(
            "host_probability",
            ascending=False
        ).head(top_n)

        top_candidates = pd.concat(
            [top_candidates, batch_top],
            ignore_index=True
        )

        top_candidates = top_candidates.drop_duplicates(
            subset="gaia_dr3_id"
        )

        top_candidates = top_candidates.sort_values(
            "host_probability",
            ascending=False
        ).head(top_n)

        print(f"Batch stars scored: {len(batch_df)}")
        print(f"Current best probability: {top_candidates['host_probability'].max():.4f}")
        print(f"Current top candidates stored: {len(top_candidates)}")

    if output_path is not None:
        top_candidates.to_csv(output_path, index=False)
        print(f"\nSaved top {top_n} candidates to: {output_path}")

    return top_candidates

In [6]:
top_100 = find_top_gaia_candidates(
    model_pipeline=xgb_pipeline,
    fetch_function=fetch_gaia,
    prepare_model_input=prepare_model_input,
    n_iterations=10,
    n_stars_per_batch=100_000,
    top_n=100,
    random_index_start=1_000_000,
    random_index_step=1_000_000,
    output_path=PROJECT_ROOT/'data'/'processed'/'top_100_gaia_candidates.csv'
)


Batch 1/10
Random index range: 1000000 - 2000000
Fetching 100000 Gaia stars for mode='prediction'...

Fetched: 70630 unique stars
Null counts:
gaia_dr3_id    0
st_teff        0
st_logg        0
st_met         0
sy_dist        0
sy_plx         0
st_rad         0
st_mass        0
st_lum         0
st_age         0
dtype: int64

Batch stars scored: 70630
Current best probability: 0.9990
Current top candidates stored: 100
Batch 2/10
Random index range: 2000000 - 3000000
Fetching 100000 Gaia stars for mode='prediction'...

Fetched: 71062 unique stars
Null counts:
gaia_dr3_id    0
st_teff        0
st_logg        0
st_met         0
sy_dist        0
sy_plx         0
st_rad         0
st_mass        0
st_lum         0
st_age         0
dtype: int64

Batch stars scored: 71062
Current best probability: 0.9996
Current top candidates stored: 100
Batch 3/10
Random index range: 3000000 - 4000000
Fetching 100000 Gaia stars for mode='prediction'...

Fetched: 71005 unique stars
Null counts:
gaia_dr3_id    

In [7]:
top_100.info()

<class 'pandas.DataFrame'>
Index: 100 entries, 0 to 88
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gaia_dr3_id       100 non-null    int64  
 1   st_teff           100 non-null    float32
 2   st_logg           100 non-null    float32
 3   st_met            100 non-null    float32
 4   sy_dist           100 non-null    float32
 5   sy_plx            100 non-null    float64
 6   st_rad            100 non-null    float32
 7   st_mass           100 non-null    float32
 8   st_lum            100 non-null    float32
 9   st_age            100 non-null    float32
 10  host_probability  100 non-null    float32
 11  candidate_label   100 non-null    str    
dtypes: float32(9), float64(1), int64(1), str(1)
memory usage: 6.6 KB


In [16]:
top_100.groupby('gaia_dr3_id')['host_probability'].max()

gaia_dr3_id
245354855808383232     0.998698
351393951913502208     0.999582
368281114781923200     0.999243
407303400929383040     0.998555
459945303249555584     0.998919
                         ...   
6232808818575221120    0.998880
6274165814582298368    0.999184
6667443252477901312    0.999520
6727293999706678528    0.999027
6764084796944951040    0.998990
Name: host_probability, Length: 100, dtype: float32

In [22]:
top = top_100.sort_values(
    "host_probability",
    ascending=False
)

top[[
    "gaia_dr3_id",
    "host_probability"
]].head(20)

,gaia_dr3_id,host_probability
0,5923445416658860160,0.999895
57,552331149296256256,0.999895
66,245354855808383232,0.999895
65,5278349845697316352,0.999895
64,4176769800317785856,0.999895
63,1807328117988735616,0.999895
62,4108337674081666688,0.999895
61,6028816284539213312,0.999895
60,2219609450418499200,0.999895
59,1333443038803929856,0.999895


In [ ]:
from astroquery.gaia import Gaia
import pandas as pd

source_id = 5923445416658860160
# Uncomment when needed. 
# Gaia.login(user="", password="")
query = f"""
SELECT
    gs.source_id,
    gs.ra,
    gs.dec,
    gs.parallax,
    gs.phot_g_mean_mag,
    gs.bp_rp,
    gs.teff_gspphot,
    gs.logg_gspphot,
    gs.mh_gspphot,
    gs.distance_gspphot,
    ap.radius_gspphot,
    ap.mass_flame,
    ap.lum_flame,
    ap.age_flame
    FROM gaiadr3.gaia_source AS gs
    LEFT JOIN gaiadr3.astrophysical_parameters AS ap
        ON gs.source_id = ap.source_id
    WHERE gs.source_id = {source_id}
"""

job = Gaia.launch_job(query)

result = job.get_results().to_pandas()

if result.empty:
    print("Star not found in Gaia DR3")
else:
    print(result)

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
             source_id         ra        dec  parallax  phot_g_mean_mag  \
0  5923445416658860160  257.27848 -55.041571  0.191114        17.493204   

      bp_rp  teff_gspphot  logg_gspphot  mh_gspphot  distance_gspphot  \
0  1.454634   4888.711914        4.6337      0.1199       1362.443726   

   radius_gspphot  mass_flame  lum_flame  age_flame  
0           0.718    0.815807   0.256523   0.200591  


In [24]:
from astroquery.simbad import Simbad

result = Simbad.query_object("Gaia DR3 5923445416658860160")

print(result)

main_id  ra dec coo_err_maj ... coo_wavelength coo_bibcode matched_id
        deg deg     mas     ...                                      
------- --- --- ----------- ... -------------- ----------- ----------
